# This is for extracting collapsed faults of each gate 
# generating the systemC code for comb and seq including the faults

In [1]:
import os

In [3]:
benchmark_name = "s5378_netlist_scanInserted"
input_transition = 0.01 #used in generation of sysC TB

In [4]:
## verilog preprocess:
file_path = benchmark_name+".v"
preVer = []
with open(file_path, "r") as file:
    for line in file:
        if "not" in line:
            outp = (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" ")
            inp = (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" ")
            line = line.replace("not ", "nand ").replace("NOT1_","NAND2_rep").replace(inp+");", inp+", "+inp+");")
            preVer.append(line)
        else:
            preVer.append(line)

directory_path = benchmark_name+"/Verilog/"
os.makedirs(directory_path, exist_ok=True)
Ver_path = os.path.join(directory_path+benchmark_name+".v")
with open(Ver_path, "w") as file:
    file.writelines(preVer)
file.close()


In [5]:
## One input gates
class gate1:
    def __init__(self, name, numOfInp, outputZN, inputA1, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [6]:
## Two input gates
class gate2:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [7]:
## Three input gates
class gate3:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n ZN = {self.outputZN}"


In [8]:
## Four input gates
class gate4:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, inputA4, gateNum):
        self.name = name
        self.gateNum = gateNum
        self.numOfInp = numOfInp
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.inputA4 = inputA4
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n A4 = {self.inputA4}\n ZN = {self.outputZN}"


In [9]:
## Four input gates
class DFF:
    def __init__(self, name, numOfInp, clk, CE, CLR, D, NbarT, PRE, Q, Si, global_reset, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.clk = clk
        self.Q = Q
        self.D = D
        self.CE = CE
        self.CLR = CLR
        self.NbarT = NbarT
        self.PRE = PRE
        self.Si = Si
        self.global_reset = global_reset
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n D = {self.D}\n Q = {self.Q}\n gate Type = {self.type}"


In [10]:
directory_path = benchmark_name+"/Verilog/"
file_path = os.path.join(directory_path+benchmark_name+".v")
i = 0
j = 0
net = []
required_gates = []
newGate = False
PIs = []
POs = []
assign_left = []
assign_right = []

numOfPI = 0
numOfPO = 0

wires=[]
wire_lines=""
with open(file_path, "r") as file:
    for line in file:
        if 'wire' in line:
            if(';' in line):
                wire_lines += line.rstrip('\n').split('wire')[1]
            else:
                wire_lines += line.rstrip('\n').split('wire')[1]
                while not(';' in line):
                    line = file.readline()  
                    wire_lines += line.rstrip('\n')
        wires = wire_lines
wires = wires.replace(" ", "").replace(";",",").split(',')
if (wires[len(wires)-1] == ""):
    wires = wires[:-1]
else: 
    wires[len(wires)-1] = wires[len(wires)-1].strip(',')
numOfwire = len(wires)
file.close()


with open(file_path, "r") as file:
    for line in file:
        if 'input' in line:
            PI= line.rstrip('\n').split('input')[1].strip(';').replace(" ", "")
            numOfPI += 1
            # print(numOfPI)
            PIs.append(PI)
            # PI_lines = line.rstrip('\n').split('input')[1]
            # ## handling enters
            # while ';' not in line:
            #     line = file.readline()
            # numOfPI = len(PI_lines.split()[1:])
            # PIs = PI_lines.replace(" ", "").strip(';').split(',')

        elif 'output' in line:
            PO= line.rstrip('\n').split('output')[1].strip(';').replace(" ", "")
            numOfPO += 1
            # print(numOfPO)
            POs.append(PO)
            # PO_lines = line.rstrip('\n').split('output')[1]
            # while ';' not in line:
            #     line = file.readline()
            #     PO_lines += line.rstrip('\n')
            # numOfPO = len(PO_lines.split()[1:])
            # POs = PO_lines.replace(" ", "").strip(';').split(',')

        elif 'dff' in line:
            # gateInps =[]
            print(line)
            newGate = True
            numOfGateInput = -1 # meand DFF
            j=j+1
            currGateName = line.split()[1].strip("(")
            print(currGateName)
            # print(line.split()[1][0:5])
            if ("DFF" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("DFF")
            line = file.readline() ##first input
            C = (line.split('(')[1].split(')')[0])
            print ("C : "+str(C))

            line = file.readline()
            CE = (line.split('(')[1].split(')')[0])
            print ("CE : "+str(CE))

            line = file.readline()
            CLR = (line.split('(')[1].split(')')[0])

            line = file.readline() ##first input
            D = (line.split('(')[1].split(')')[0])
            print ("D : "+str(D))

            line = file.readline()
            NbarT = (line.split('(')[1].split(')')[0])

            line = file.readline()
            PRE = (line.split('(')[1].split(')')[0])

            line = file.readline() ##first input
            Q = (line.split('(')[1].split(')')[0])
            print ("Q : "+str(Q))

            line = file.readline()
            Si = (line.split('(')[1].split(')')[0])

            line = file.readline()
            global_reset = (line.split('(')[1].split(')')[0])
            
            gateType = "dff"
            
        
        elif 'NAND' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NAND" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NAND")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NAND"+str(numOfGateInput)+"_X1"
            # if (numOfGateInput == 1):
            #     net.append(gate1(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 2):
            #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 3):
            #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOR' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NOR" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NOR")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NOR"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # if (line.split()[1][0:4] not in required_gates):
            #     required_gates.append(line.split()[1][0:4])
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            # # if (numOfGateInput == 2):
            # #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # # if (numOfGateInput == 3):
            # #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOT' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("INV" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("INV")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "INV"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # currGateName = "INV"
            # if (currGateName not in required_gates):
            #     required_gates.append(currGateName)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1

        elif 'assign' in line:
            # print(line.split())
            assign_left.append(line.split()[1])
            assign_right.append(line.split()[3].strip(';'))

        # elif ' or' in line:
        #     # print(line)
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "OR"+str(numOfGateInput)
        #     # currGateName = line.split()[1][0:3]
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        #     # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1


        # elif 'and' in line:
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "AND"+str(numOfGateInput)
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        else:
            newGate = False

        if (newGate):
            if (numOfGateInput == 1):
                # net.append(gate1(currGateName, 1, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), i))
                net.append(gate1(currGateName, 1, gateOutp, gateInps[0], i, gateType))

            elif (numOfGateInput == 2):
                # net.append(gate2(currGateName, 2,(line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
                net.append(gate2(currGateName, 2, gateOutp, gateInps[0], gateInps[1], i, gateType))

            elif (numOfGateInput == 3):
                net.append(gate3(currGateName, 3, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
            elif (numOfGateInput == 4):
                net.append(gate4(currGateName, 4, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[4].strip(" "), i))
            elif (numOfGateInput == -1): # DFF
                net.append(DFF(currGateName, -1, C, CE, CLR, D, NbarT, PRE, Q, Si, global_reset, j, gateType))
file.close()

 dff DFF_0(

DFF_0
C : CK
CE : 1'b1
D : n2897gat
Q : n672gat
 dff DFF_1(

DFF_1
C : CK
CE : 1'b1
D : n3069gat
Q : II729
 dff DFF_10(

DFF_10
C : CK
CE : 1'b1
D : n3065gat
Q : II368
 dff DFF_100(

DFF_100
C : CK
CE : 1'b1
D : n3053gat
Q : II2425
 dff DFF_101(

DFF_101
C : CK
CE : 1'b1
D : n2613gat
Q : n2506gat
 dff DFF_102(

DFF_102
C : CK
CE : 1'b1
D : n1625gat
Q : n1834gat
 dff DFF_103(

DFF_103
C : CK
CE : 1'b1
D : n1626gat
Q : n1767gat
 dff DFF_104(

DFF_104
C : CK
CE : 1'b1
D : n1442gat
Q : n2084gat
 dff DFF_105(

DFF_105
C : CK
CE : 1'b1
D : n2482gat
Q : n1787gat
 dff DFF_106(

DFF_106
C : CK
CE : 1'b1
D : n2557gat
Q : n1785gat
 dff DFF_107(

DFF_107
C : CK
CE : 1'b1
D : n2487gat
Q : n1633gat
 dff DFF_108(

DFF_108
C : CK
CE : 1'b1
D : n2532gat
Q : n1884gat
 dff DFF_109(

DFF_109
C : CK
CE : 1'b1
D : n2486gat
Q : n1631gat
 dff DFF_11(

DFF_11
C : CK
CE : 1'b1
D : n3067gat
Q : II359
 dff DFF_110(

DFF_110
C : CK
CE : 1'b1
D : n2353gat
Q : n1989gat
 dff DFF_111(

DFF_111
C : CK
CE :

In [11]:
print(assign_left)
print(assign_right)


['So', 'II196', 'II203', 'II210', 'II2813', 'II3235', 'II3318', 'II3394', 'II3461', 'II3509', 'II3951', 'II3954', 'II4117', 'II4122', 'II4222', 'II4227', 'II4482', 'II4485', 'II4489', 'II4492', 'II4496', 'II4499', 'II4558', 'II4630', 'II4633', 'II4660', 'n1018gat', 'n1019gat', 'n1020gat', 'n1025gat', 'n1026gat', 'n1034gat', 'n1035gat', 'n1044gat', 'n1045gat', 'n1055gat', 'n1056gat', 'n1057gat', 'n1067gat', 'n1068gat', 'n1071gat', 'n1072gat', 'n1079gat', 'n1080gat', 'n1084gat', 'n1085gat', 'n1118gat', 'n1120gat', 'n1121gat', 'n1134gat', 'n1135gat', 'n1147gat', 'n1148gat', 'n1150gat', 'n1189gat', 'n1190gat', 'n1191gat', 'n1197gat', 'n1206gat', 'n1207gat', 'n1208gat', 'n1221gat', 'n1222gat', 'n1223gat', 'n1226gat', 'n1233gat', 'n1234gat', 'n1235gat', 'n1240gat', 'n1241gat', 'n1269gat', 'n1282gat', 'n1293gat', 'n1294gat', 'n1297gat', 'n1298gat', 'n1312gat', 'n1316gat', 'n1328gat', 'n1340gat', 'n1350gat', 'n1382gat', 'n1394gat', 'n1456gat', 'n1462gat', 'n148gat', 'n1496gat', 'n1508gat', 'n1

In [12]:
print(net[3])

 gateNum = 4
 gateName = _2421_
 A1 = _1900_
 ZN = _2014_
 gate Type = INV1_X1


In [13]:
# num of gates
len(net)

1505

In [14]:
required_gates

['INV', 'NOR', 'NAND', 'DFF']

In [31]:
print(len(PIs))
POs

39


['So',
 'n3104gat',
 'n3105gat',
 'n3106gat',
 'n3107gat',
 'n3108gat',
 'n3109gat',
 'n3110gat',
 'n3111gat',
 'n3112gat',
 'n3113gat',
 'n3114gat',
 'n3115gat',
 'n3116gat',
 'n3117gat',
 'n3118gat',
 'n3119gat',
 'n3120gat',
 'n3121gat',
 'n3122gat',
 'n3123gat',
 'n3124gat',
 'n3125gat',
 'n3126gat',
 'n3127gat',
 'n3128gat',
 'n3129gat',
 'n3130gat',
 'n3131gat',
 'n3132gat',
 'n3133gat',
 'n3134gat',
 'n3135gat',
 'n3136gat',
 'n3137gat',
 'n3138gat',
 'n3139gat',
 'n3140gat',
 'n3141gat',
 'n3142gat',
 'n3143gat',
 'n3144gat',
 'n3145gat',
 'n3146gat',
 'n3147gat',
 'n3148gat',
 'n3149gat',
 'n3150gat',
 'n3151gat',
 'n3152gat']

In [16]:
# read the faults list, store the content in a dectionary, key: net, value: SA
directory_path = "s5378_faultlist.flt"
# file_path = os.path.join(directory_path+benchmark_name+".v")
faultDict = {}
file_path = directory_path
with open(file_path, "r") as file:
    for line in file:  
        wire, SA = line.split()
        print(wire)
        print(SA)      
        if ( wire in faultDict):
            SAs=[]
            SAs.append(faultDict[wire])
            SAs.append(SA)
            faultDict[wire] = SAs
        else:
            faultDict[wire] = SA.strip("\n")

print(faultDict)

n3083gat
s@0
n3083gat
s@1
n3084gat
s@0
n3084gat
s@1
n3085gat
s@0
n3085gat
s@1
n3086gat
s@0
n3086gat
s@1
n3087gat
s@0
n3087gat
s@1
n3088gat
s@0
n3088gat
s@1
n3089gat
s@0
n3089gat
s@1
n3093gat
s@0
n3093gat
s@1
n3095gat
s@0
n3095gat
s@1
n3104gat
s@1
n3104gat
s@0
n3151gat
s@1
n3151gat
s@0
n3148gat
s@0
n3148gat
s@1
n3147gat
s@0
n3147gat
s@1
n3146gat
s@0
n3146gat
s@1
n3114gat
s@1
n3114gat
s@0
n3113gat
s@1
n3113gat
s@0
n3105gat
s@1
n3105gat
s@0
n3150gat
s@1
n3150gat
s@0
n3111gat
s@1
n3111gat
s@0
n3110gat
s@1
n3110gat
s@0
n3108gat
s@1
n3108gat
s@0
n3109gat
s@1
n3109gat
s@0
n3144gat
s@1
n3144gat
s@0
n3143gat
s@1
n3143gat
s@0
n3125gat
s@1
n3125gat
s@0
n3124gat
s@1
n3124gat
s@0
n3123gat
s@1
n3123gat
s@0
n3122gat
s@1
n3122gat
s@0
n3121gat
s@1
n3121gat
s@0
n3120gat
s@1
n3120gat
s@0
n3119gat
s@1
n3119gat
s@0
n3118gat
s@1
n3118gat
s@0
_2921_.Y
s@0
_2921_.Y
s@1
_2922_.Y
s@0
_2922_.B
s@0
_2922_.A
s@0
_2922_.Y
s@1
_2923_.Y
s@0
_2923_.Y
s@1
_2925_.Y
s@1
_2925_.B
s@1
_2925_.Y
s@0
_2924_.B
s@1
_2924_.A
s@1

In [17]:
faultDict

{'n3083gat': ['s@0', 's@1'],
 'n3084gat': ['s@0', 's@1'],
 'n3085gat': ['s@0', 's@1'],
 'n3086gat': ['s@0', 's@1'],
 'n3087gat': ['s@0', 's@1'],
 'n3088gat': ['s@0', 's@1'],
 'n3089gat': ['s@0', 's@1'],
 'n3093gat': ['s@0', 's@1'],
 'n3095gat': ['s@0', 's@1'],
 'n3104gat': ['s@1', 's@0'],
 'n3151gat': ['s@1', 's@0'],
 'n3148gat': ['s@0', 's@1'],
 'n3147gat': ['s@0', 's@1'],
 'n3146gat': ['s@0', 's@1'],
 'n3114gat': ['s@1', 's@0'],
 'n3113gat': ['s@1', 's@0'],
 'n3105gat': ['s@1', 's@0'],
 'n3150gat': ['s@1', 's@0'],
 'n3111gat': ['s@1', 's@0'],
 'n3110gat': ['s@1', 's@0'],
 'n3108gat': ['s@1', 's@0'],
 'n3109gat': ['s@1', 's@0'],
 'n3144gat': ['s@1', 's@0'],
 'n3143gat': ['s@1', 's@0'],
 'n3125gat': ['s@1', 's@0'],
 'n3124gat': ['s@1', 's@0'],
 'n3123gat': ['s@1', 's@0'],
 'n3122gat': ['s@1', 's@0'],
 'n3121gat': ['s@1', 's@0'],
 'n3120gat': ['s@1', 's@0'],
 'n3119gat': ['s@1', 's@0'],
 'n3118gat': ['s@1', 's@0'],
 '_2921_.Y': ['s@0', 's@1'],
 '_2922_.Y': ['s@0', 's@1'],
 '_2922_.B': '

In [18]:
newFaultDict = {}
POFaults=0
PIFaults=0
DFFFaults=0
Faults=0

for item in faultDict:
    if(item.startswith("DFF")):
        print("FOUND DFF")
        if(len(faultDict[item])>2):
                DFFFaults+=1
        else:
            DFFFaults+=len(faultDict[item])
    else:
        if (item  in POs): ## the wire is a PO add the fault to gate OutPut
            for g in net:
                if(g.numOfInp!=-1): ## LEAVE OUT DFFS
                    if(g.outputZN == assign_right[assign_left.index(item)]):
                        new_element_key = g.name+".Y"
                        new_element_value = faultDict[item]
                        POFaults+=1
                        if(len(faultDict[item])>2):
                            Faults+=1
                        else:
                            Faults+=len(faultDict[item])
                        newFaultDict[item] = faultDict[item]
                        newFaultDict[new_element_key] = new_element_value ## add the faults to the gate OP
        elif(item in PIs):
                print("FOUND PI")
                print(faultDict[item])
                print(len(faultDict[item]))
                if(len(faultDict[item])>2):
                    PIFaults+=1
                else:
                    PIFaults+=len(faultDict[item])
        else:
            if(len(faultDict[item])>2):
                Faults+=1
            else:
                Faults+=len(faultDict[item])
            newFaultDict[item] = faultDict[item]


FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
s@0
3
FOUND PI
s@0
3
FOUND DFF
FOUND DFF
FOUND DFF
FOUND PI
s@0
3
FOUND PI
s@1
3
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND PI
['s@0', 's@1']
2
FOUND DFF
FOUND DFF
FOUND DFF
FOUND DFF
FOUND DFF
FOUND DFF
FOUND DFF
FOUND DF

In [19]:
print(PIFaults)
print(POFaults)
print(DFFFaults)
print(Faults)



68
40
220
1902


In [20]:
# # del my_dict['b']
# newFaultDict = {}
# for item in faultDict:
#     add = True
    
#     if (item  in POs): ## the wire is a PO add the fault to gate OutPut
#         print("FOUND PO")
#         for g in net:
#             if(g.numOfInp!=-1): ## LEAVE OUT DFFS
#                 if(g.outputZN == assign_right[assign_left.index(item)]):
#                     new_element_key = g.name+".Y"
#                     new_element_value = faultDict[item]
#                     newFaultDict[new_element_key] = new_element_value ## add the faults to the gate OP
#     else: ## not an output but could be an input
#         gateName = item.split('.')[0] ## gate name
#         for g in net:
#             if(g.name == gateName and g.numOfInp==2): ## if it is a two input gate
#                 if ((g.inputA1 in assign_left)):
#                     if(assign_right[assign_left.index(g.inputA1)] in PIs):
#                         print("cought PI")
#                         print(gateName)
#                         print(g.inputA1)
#                         add = False
#                         # break
#                 if((g.inputA2 in assign_left)):
#                     if(assign_right[assign_left.index(g.inputA2)] in PIs):
#                         print("cought PI")
#                         print(gateName)
#                         print(g.inputA1)
#                         add = False
#                         # break
#             if(g.name == gateName and g.numOfInp==1):
#                 if ((g.inputA1 in assign_left)):
#                     if(assign_right[assign_left.index(g.inputA1)] in PIs):
#                         print("cought PI")
#                         print(gateName)
#                         print(g.inputA1)
#                         add = False
#                         # break
                
#         if(add):
#             newFaultDict[item] = faultDict[item]
#             # count+=len(faultDict[item])


In [21]:
newFaultDict

{'n3104gat': ['s@1', 's@0'],
 '_3686_.Y': ['s@1', 's@0'],
 'n3151gat': ['s@1', 's@0'],
 '_3329_.Y': ['s@1', 's@0'],
 'n3148gat': ['s@0', 's@1'],
 '_2519_.Y': ['s@0', 's@1'],
 'n3147gat': ['s@0', 's@1'],
 '_2520_.Y': ['s@0', 's@1'],
 'n3146gat': ['s@0', 's@1'],
 '_2990_.Y': ['s@0', 's@1'],
 'n3114gat': ['s@1', 's@0'],
 '_3667_.Y': ['s@1', 's@0'],
 'n3113gat': ['s@1', 's@0'],
 '_3668_.Y': ['s@1', 's@0'],
 'n3105gat': ['s@1', 's@0'],
 '_3680_.Y': ['s@1', 's@0'],
 'n3150gat': ['s@1', 's@0'],
 '_3331_.Y': ['s@1', 's@0'],
 'n3111gat': ['s@1', 's@0'],
 '_3669_.Y': ['s@1', 's@0'],
 'n3110gat': ['s@1', 's@0'],
 '_3670_.Y': ['s@1', 's@0'],
 'n3108gat': ['s@1', 's@0'],
 '_3672_.Y': ['s@1', 's@0'],
 'n3109gat': ['s@1', 's@0'],
 '_3671_.Y': ['s@1', 's@0'],
 'n3144gat': ['s@1', 's@0'],
 '_3444_.Y': ['s@1', 's@0'],
 'n3143gat': ['s@1', 's@0'],
 '_3445_.Y': ['s@1', 's@0'],
 'n3125gat': ['s@1', 's@0'],
 '_3650_.Y': ['s@1', 's@0'],
 'n3124gat': ['s@1', 's@0'],
 '_3652_.Y': ['s@1', 's@0'],
 'n3123gat': [

In [22]:
gateFault_dict = {}
line = []
tot =0
for g in net:
    if(g.numOfInp!=-1):
        faults = ""
        print("new gate "+g.name)
        key = g.name
        # if ((g.name+".Y" not in newFaultDict) and (g.name+".A" not in newFaultDict) and):

        if (g.name+".Y" in newFaultDict):
            if ("s@0" in newFaultDict[g.name+".Y"]):
                line.append("s@0 "+g.name+".Y\n")
                faults +="1"
                tot+=1
            else:
                faults +="0"
            if ("s@1" in newFaultDict[g.name+".Y"]):
                line.append("s@1 "+g.name+".Y\n")
                faults +="1"
                tot+=1

            else:
                faults +="0"
        else:
            faults +="00"

        if (g.name+".A" in newFaultDict):
        
            if ("s@0" in newFaultDict[g.name+".A"]):
                line.append("s@0 "+g.name+".A\n")
                faults +="1"
                tot+=1
            else:
                faults +="0"
            if ("s@1" in newFaultDict[g.name+".A"]):
                line.append("s@1 "+g.name+".A\n")
                faults +="1"
                tot+=1
            else:
                faults +="0"
        else:
            faults +="00"

        if (g.name+".B" in newFaultDict):
            if ("s@0" in newFaultDict[g.name+".B"]):
                line.append("s@0 "+g.name+".B\n")
                faults +="1"
                tot+=1 
            else:
                faults +="0"
            if ("s@1" in newFaultDict[g.name+".B"]):
                line.append("s@1 "+g.name+".B\n")
                faults +="1"
                tot+=1
            else:
                faults +="0"
        else:
            faults +="00"
            
        if (faults == ""):
            faults = "0000"
            print("DAMN U"+g.name)
            
        print(key)
        if(int(faults, 2) != 0):
            gateFault_dict[key] = int(faults, 2)
        if(key == "_450_"):
                    print(gateFault_dict[key])
    # print(faults)
        

    # print(g.inputA2)


new gate _2418_
_2418_
new gate _2419_
_2419_
new gate _2420_
_2420_
new gate _2421_
_2421_
new gate _2422_
_2422_
new gate _2423_
_2423_
new gate _2424_
_2424_
new gate _2425_
_2425_
new gate _2426_
_2426_
new gate _2427_
_2427_
new gate _2428_
_2428_
new gate _2429_
_2429_
new gate _2430_
_2430_
new gate _2431_
_2431_
new gate _2432_
_2432_
new gate _2433_
_2433_
new gate _2434_
_2434_
new gate _2435_
_2435_
new gate _2436_
_2436_
new gate _2437_
_2437_
new gate _2438_
_2438_
new gate _2439_
_2439_
new gate _2440_
_2440_
new gate _2441_
_2441_
new gate _2442_
_2442_
new gate _2443_
_2443_
new gate _2444_
_2444_
new gate _2445_
_2445_
new gate _2446_
_2446_
new gate _2447_
_2447_
new gate _2448_
_2448_
new gate _2449_
_2449_
new gate _2450_
_2450_
new gate _2451_
_2451_
new gate _2452_
_2452_
new gate _2453_
_2453_
new gate _2454_
_2454_
new gate _2455_
_2455_
new gate _2456_
_2456_
new gate _2457_
_2457_
new gate _2458_
_2458_
new gate _2459_
_2459_
new gate _2460_
_2460_
new gate _2

In [23]:
tot

1894

In [24]:
gateFault_dict

{'_2420_': 48,
 '_2421_': 48,
 '_2422_': 48,
 '_2423_': 48,
 '_2424_': 48,
 '_2425_': 48,
 '_2426_': 32,
 '_2427_': 48,
 '_2428_': 48,
 '_2429_': 48,
 '_2430_': 16,
 '_2431_': 16,
 '_2432_': 48,
 '_2433_': 48,
 '_2434_': 48,
 '_2435_': 32,
 '_2436_': 48,
 '_2437_': 48,
 '_2438_': 32,
 '_2439_': 40,
 '_2440_': 48,
 '_2441_': 48,
 '_2442_': 32,
 '_2443_': 48,
 '_2444_': 32,
 '_2445_': 34,
 '_2446_': 48,
 '_2447_': 42,
 '_2448_': 48,
 '_2449_': 42,
 '_2450_': 5,
 '_2451_': 32,
 '_2452_': 48,
 '_2453_': 16,
 '_2454_': 48,
 '_2455_': 32,
 '_2456_': 32,
 '_2457_': 32,
 '_2458_': 42,
 '_2459_': 48,
 '_2460_': 42,
 '_2461_': 5,
 '_2462_': 32,
 '_2463_': 48,
 '_2464_': 32,
 '_2465_': 48,
 '_2466_': 40,
 '_2467_': 42,
 '_2469_': 48,
 '_2470_': 53,
 '_2471_': 58,
 '_2472_': 16,
 '_2473_': 17,
 '_2474_': 16,
 '_2475_': 32,
 '_2476_': 24,
 '_2477_': 17,
 '_2478_': 48,
 '_2479_': 32,
 '_2480_': 32,
 '_2481_': 48,
 '_2482_': 48,
 '_2483_': 48,
 '_2484_': 48,
 '_2485_': 48,
 '_2486_': 32,
 '_2487_': 3

In [25]:
directory_path = benchmark_name+"/faults/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"faultlist.flt")
with open(sysC_output_file_path, "w") as file:
    file.writelines(line)
file.close()

In [34]:
## systemC output
sysC_H = []
# for required_gate in required_gates:
#     sysC_H.append("#include \""+required_gate+"_X1.h\"\n")
sysC_H.append("#include \"NAND2_X1.h\"\n")
sysC_H.append("#include \"NOR2_X1.h\"\n")
sysC_H.append("#include \"INV1_X1.h\"\n")
sysC_H.append("#include <iostream>\n")
sysC_H.append("#include <fstream>\n")
sysC_H.append("#include <string>\n")
sysC_H.append("#include <vector>\n")
sysC_H.append("#include <set>\n")
sysC_H.append("#include <map>\n\n\n")


sysC_H.append("extern std::map<int, GATES_X1*> module_map;\n")


sysC_H.append("SC_MODULE("+benchmark_name+")\n{\n")
sysC_H.append("\tsc_in <fs_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_H.append(PIs[PI_idx]+", ")
sysC_H.append(PIs[len(PIs)-1]+";\n")

sysC_H.append("\tsc_out <fs_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_H.append(POs[PO_idx]+", ")
sysC_H.append(POs[len(POs)-1]+";\n")

sysC_H.append("\tsc_signal <fs_logic> ")
for wire_idx in range(len(wires)-1):
    if((not(wires[wire_idx] in PIs)) and (not(wires[wire_idx] in POs))):
        sysC_H.append(wires[wire_idx]+", ")
sysC_H.append(wires[len(wires)-1]+";\n")

sysC_H.append("\tsc_in<sc_logic> endSim; \n")
sysC_H.append("\tsc_in<sc_logic> newTV; \n")
sysC_H.append("\tsc_uint<32> counter; \n")

sysC_H.append("\n\tint numOfGates;\n\tint totalObservedCombs;\n\tdouble GIC_Coverage;\n\n")


# sysC_H.append("\n\tint numOfGates;\n\tdouble t;\n\tdouble outLoad;\n\n")

i = 1
for g in net:
    # sysC_H.append("\t"+g.name+"_X1* "+g.name+"_Gate"+str(i)+";\n")
    sysC_H.append("\t"+g.type+"* "+g.name+"_Gate"+str(i)+";\n")
    i = i+1

sysC_H.append("\n\tSC_CTOR("+benchmark_name+")\n\t{\n\t\tnumOfGates = "+str(len(net))+";\n\n")



i = 1
for g in net:
    if (g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.type+"(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    else:
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+" = new "+g.type+"(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")

    if (g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->C"+"("+g.clk+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CE"+"(sc_logic_1_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CLR"+"("+g.CLR+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->D"+"("+g.D+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->NbarT"+"("+g.NbarT+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->PRE"+"(sc_logic_0_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Si"+"("+g.Si+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->global_reset"+"(sc_logic_0_signal);\n")

    if (g.numOfInp == 1):
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->CP(takeCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->ldCP(loadCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->NbarT(fpMode);\n")


    if (g.numOfInp == 2):
        if (g.name.startswith("XOR")): ## XOR has only 2
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->B"+"("+g.inputA2+");\n")
        else:    
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->CP(takeCP);\n")
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->ldCP(loadCP);\n")
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->NbarT(fpMode);\n")
    if (g.numOfInp == 3):
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->CP(takeCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->ldCP(loadCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->NbarT(fpMode);\n")
    if (g.numOfInp == 4):
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->A4"+"("+g.inputA4+");\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->CP(takeCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->ldCP(loadCP);\n")
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->NbarT(fpMode);\n")
    if(g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Q"+"("+g.Q+");\n\n")
    else:
        sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->ZN"+"("+g.outputZN+");\n")
    
    if(g.name in gateFault_dict):
        if(g.numOfInp == -1):
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->faults"+"="+str(gateFault_dict[g.name])+";\n")
        else:
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->faults"+"="+str(gateFault_dict[g.name])+";\n")
    else:
        if(g.numOfInp == -1):
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->faults"+"=0;\n")
        else:
            sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->faults"+"=0;\n")
    if(g.numOfInp == -1):
        sysC_H.append("\t\tmodule_map["+str(i)+"] = "+g.name+"_Gate"+str(i)+";\n\n")
    else:
        sysC_H.append("\t\tmodule_map["+str(i)+"] = "+g.name+"Gate"+str(i)+";\n\n")

        


    # if (g.outputZN in POs):
    #     sysC_H.append("\t\t"+g.name+"Gate"+str(i)+"->load_c"+" = outLoad;\n")
    # else:
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c = 0;\n")
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->aged_time = t;\n\n")
    i = i+1

sysC_H.append("\t\tcout << \"all gates are instantiated \" << numOfGates << \"\\n\";\n")

sysC_H.append("\t\tSC_THREAD(assignments);\n")
# //sensitivity
j=0
unique_sens_list = list(set(assign_right))
sysC_H.append("\t\tsensitive")
for j in range(len(unique_sens_list)):
    sysC_H.append(" << "+unique_sens_list[j])
sysC_H.append(";\n")
sysC_H.append("\t\tSC_METHOD(GIC_Coverage_Calculator);\n")
sysC_H.append("\t\tsensitive << endSim<< newTV;\n\n\t}\n")

sysC_H.append("\tvoid ini();\n\tvoid assignments();\n\tvoid GIC_Coverage_Calculator();\n")

# j = 1
# for required_gate_idx in range(len(required_gates)):
#     sysC_H.append("\tvoid notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
#     sysC_H.append(required_gates[required_gate_idx]+"_X1 *gate"+");\n")
#     # sysC_H.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+");\n};\n")
#     j=j+1
sysC_H.append("\n};\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"netlist.h")
with open(sysC_output_file_path, "w") as file:
    file.writelines(sysC_H)
file.close()

In [16]:
sysC_C = []
sysC_C.append("#include \"netlist.h\"\n#include <cmath>\n")
sysC_C.append("std::ofstream GIC_logFile(\"GIC_logFile.txt\");\n\n")
sysC_C.append("void "+benchmark_name+"::assignments()\n{\n\twhile (true)\n\t{\n")

m=0
for m in range(len(assign_left)):
    sysC_C.append("\t\t"+assign_left[m]+".write("+assign_right[m]+");\n")

sysC_C.append("\n\t\twait();\n\t}\n}\n\n")

sysC_C.append("void "+benchmark_name+"::GIC_Coverage_Calculator()\n{\n")
sysC_C.append("\ttotalObservedCombs =\n")
n=0
for n in range(len(net)):
    if n==len(net)-1:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs;\n")
    else:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs +\n")

sysC_C.append("\tcout << \"END OF SIM=> Total Observed Combs = \" << totalObservedCombs << \"\\n\";\n")
sysC_C.append("\tcout << \"GIC Coverage = \" << GIC_Coverage << \"\\n\";\n\n}")



directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_C_output_file_path = os.path.join(directory_path+"netlist.cpp")
with open(sysC_C_output_file_path, "w") as file:
    file.writelines(sysC_C)
file.close()

In [17]:

sysC_TB_H = []
sysC_TB_H.append("#include \"netlist.h\"\n#include <fstream>\n\nSC_MODULE("+benchmark_name+"_TB)\n{\n\n")
sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_TB_H.append("testData"+str(PI_idx+1)+", ")
sysC_TB_H.append("testData"+str(len(PIs))+";\n")

sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_TB_H.append("testRes"+str(PO_idx+1)+", ")
sysC_TB_H.append("testRes"+str(len(POs))+";\n")

sysC_TB_H.append("\n\tsc_signal<sc_logic> reset, clock, end, newTV;\n\t"+benchmark_name+"* UUT;\n\n")
sysC_TB_H.append("\tstd::vector<std::string> testVecs;\n")

sysC_TB_H.append("\tSC_CTOR("+benchmark_name+"_TB)\n\t{\n\n\t\ttestVecs = read_testPtr (\"testPatterns.txt\");\n\t\tUUT = new "+benchmark_name+"(\""+benchmark_name+"_instance\");\n")
p=1
for PI in PIs:
    sysC_TB_H.append("\t\tUUT->"+PI.strip(" ")+"(testData"+str(p)+");\n")
    p=p+1

o=1
for PO in POs:
    sysC_TB_H.append("\t\tUUT->"+PO.strip(" ")+"(testRes"+str(o)+");\n")
    o=o+1

sysC_TB_H.append("\t\tUUT->endSim(end);\n")
sysC_TB_H.append("\t\tUUT->newTV(newTV);\n")
sysC_TB_H.append("\n\t\tSC_THREAD(testPtr);\n\t\tSC_THREAD(endOfSim);\n\t}\n")

sysC_TB_H.append("\n\tvoid endOfSim();\n\tvoid testPtr();\n\tstd::vector<std::string> read_testPtr(std::string filename);\n")
sysC_TB_H.append("};\n")




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_H_output_file_path = os.path.join(directory_path+"TB.h")
with open(sysC_TB_H_output_file_path, "w") as file:
    file.writelines(sysC_TB_H)
file.close()

In [18]:
sysC_TB_C = []
sysC_TB_C.append("#include \"TB.h\"\n\n")
sysC_TB_C.append("void "+benchmark_name+"_TB::testPtr()\n{\n\twhile (true)\n\t{\n")

m=1
for PI in PIs:
    # sysC_TB_C.append("void powerGatesNetlistTB::testData"+str(m)+"Waveform()\n{\n\twhile (true)\n{\n")
    sysC_TB_C.append("\t\ttestData"+str(m)+".write(SC_LOGIC_0);\n")
    m=m+1

sysC_TB_C.append("\t\twait(1000, SC_NS);\n")
sysC_TB_C.append("\t\tfor (int testVecidx = 0; testVecidx < testVecs.size(); testVecidx++)\n\t\t\t{\n")
s=0
for s in range(len(PIs)):
    sysC_TB_C.append("\t\t\tif (testVecs[testVecidx]["+str(s)+"] == '0')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_0);\n\t\t\telse if (testVecs[testVecidx]["+str(s)+"] == '1')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_1);\n\n")
sysC_TB_C.append("\t\twait(15000, SC_NS);\n\t\tnewTV.write(SC_LOGIC_1);\n\t\twait(0, SC_NS);\n\t\t}\n\t\twait();\n\t}\n}\n")

sysC_TB_C.append("std::vector<std::string> "+benchmark_name+"_TB::read_testPtr(std::string filename)\n{\n")
sysC_TB_C.append("\tstd::vector<std::string> selected_testVec;\n\tstd::ifstream file(filename);\n\tif (!file.is_open()) {\n\t\tstd::cerr << \"Error opening file!\" << std::endl;\t\n\t}\n\tstd::string line;\n\tstd::vector<std::string> lines;\n\twhile (std::getline (file, line))\n\t{\n\t\tlines.push_back(line);\n\t}\n\treturn lines;\n}")

sysC_TB_C.append("\nvoid "+benchmark_name+"_TB::endOfSim()\n{\n\twhile (true)\n{\n\t\tend.write(SC_LOGIC_0);\n\t\twait(3900000, SC_NS);\n\t\tend.write(SC_LOGIC_1);\n\t\twait(15000, SC_NS);\n\t\twait();\n\t}\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_C_output_file_path = os.path.join(directory_path+"TB.cpp") 
with open(sysC_TB_C_output_file_path, "w") as file:
    file.writelines(sysC_TB_C)
file.close()

In [19]:
sysC_sim_C = []
sysC_sim_C.append("#include \"TB.h\"\n#include <iostream>\n#include <fstream> \n\nint sc_main(int argc, char** argv)\n{\n\t"+benchmark_name+"_TB* TOP = new "+benchmark_name+"_TB(\"netlistSimulationTB_instance\");")
sysC_sim_C.append("\n\tsc_start(400000, SC_NS);\n\treturn 0;\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_sim_C_output_file_path = os.path.join(directory_path+"simulation.cpp") 
with open(sysC_sim_C_output_file_path, "w") as file:
    file.writelines(sysC_sim_C)
file.close()